Bản port Python của app **Valo Upscale** (Real-ESRGAN cho ảnh/video).

**Chuẩn bị (2 việc):**
1. `Runtime → Change runtime type → Hardware accelerator = T4 GPU` → *Save*.
2. Ô ngay dưới: dán link repo GitHub của bạn vào `REPO_URL` (đã điền sẵn repo của bạn — chỉ đổi khi bạn fork sang tài khoản khác).

Sau đó bấm **Runtime → Run all**. Notebook sẽ: kiểm tra GPU → clone repo → cài thư viện → chạy thử ảnh/video mẫu → cho bạn upload file hoặc lấy file từ Google Drive → nén kết quả để tải về. Cuối notebook có ô mở giao diện web Gradio.


In [ ]:
#@title 1) Cấu hình repo
REPO_URL = "https://github.com/hoannguyen171/UPSCALEVIDEO.git"  #@param {type:"string"}
REPO_DIR = "/content/valo-upscale-gpu"  #@param {type:"string"}
print("Repo :", REPO_URL)
print("Thư mục:", REPO_DIR)

In [ ]:
#@title 2) Kiểm tra GPU của Colab
!nvidia-smi || echo "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"
import platform, sys
print(platform.platform())
print(sys.version)

In [ ]:
import os, shutil, subprocess

if "CHANGE_ME" in REPO_URL:
    raise SystemExit("⚠️ Hãy sửa REPO_URL ở ô 1 thành link repo GitHub của bạn trước khi chạy.")

#@title 3) Clone repo + cài thư viện
import os, shutil, subprocess

if not shutil.which("ffmpeg"):
    print("Đang cài ffmpeg...")
    !apt-get -qq install -y ffmpeg

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True).stdout)
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
#@title 4) Kiểm tra môi trường
import subprocess, torch, valo_upscale

print("valo-upscale-gpu", valo_upscale.__version__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])
!python -m valo_upscale.cli --list-models

## 5) Chạy thử với file mẫu trong repo

Ô dưới upscale `samples/sample_lowres.png` (240×135) lên **4x** — lần chạy đầu sẽ tự tải weight
`RealESRGAN_x4plus.pth` (~64 MB) từ GitHub Releases.

In [ ]:
#@title 5a) Ảnh mẫu: 4x bằng Real-ESRGAN
!python -m valo_upscale.cli samples/sample_lowres.png -s 4

from pathlib import Path
from IPython.display import display, Image as IPyImage
before = Path("samples/sample_lowres.png")
after = max(Path("outputs").glob("*_upscaled_4x_*.png"), key=lambda p: p.stat().st_mtime)
print(f"Trước: {before.name} | Sau: {after.name} ({after.stat().st_size // 1024} KB)")
display(IPyImage(str(before), width=420))
display(IPyImage(str(after), width=420))

In [ ]:
#@title 5b) Video mẫu: 2x, giữ nguyên audio
!python -m valo_upscale.cli samples/sample_clip.mp4 -s 2

from pathlib import Path
from IPython.display import Video, display
out = max(Path("outputs").glob("*_upscaled_2x_*.mp4"), key=lambda p: p.stat().st_mtime)
print(f"{out} — {out.stat().st_size // 1024} KB")
display(Video(str(out), embed=True, width=480))

## 6) Upscale file của bạn

Ô dưới mở hộp thoại chọn file (chọn được nhiều file). Ví dụ muốn phóng 8x hoặc đổi model thì sửa
tham số trong lệnh: `-s 8 -n realesrgan-x4plus-anime`.

In [ ]:
#@title 6a) Tải file lên và chạy
from google.colab import files
import subprocess

uploaded = files.upload()  # chọn ảnh (.jpg .png) hoặc video (.mp4 .mov ...)
for name in uploaded:
    print("=" * 70)
    print("Xử lý:", name)
    subprocess.run(["python", "-m", "valo_upscale.cli", name, "-s", "4", "--tile", "400"], check=False)

In [ ]:
#@title 6b) Xem kết quả vừa tạo
from pathlib import Path
from IPython.display import Video, Image as IPyImage, display

results = sorted(Path("outputs").glob("*_upscaled_*"), key=lambda p: p.stat().st_mtime)[-6:]
for path in results:
    print(f"{path.name} — {path.stat().st_size / 1024:.0f} KB")
    display(Video(str(path), embed=True, width=420) if path.suffix == ".mp4" else IPyImage(str(path), width=420))

In [ ]:
#@title 6c) Nén toàn bộ kết quả và tải về máy
import shutil
from pathlib import Path
from google.colab import files

zip_path = shutil.make_archive("/content/ket_qua_upscale", "zip", root_dir="outputs")
print(zip_path, round(Path(zip_path).stat().st_size / 1048576, 2), "MB")
files.download(zip_path)


## 7) Lấy file trực tiếp từ Google Drive (tuỳ chọn)

Hữu ích khi file nằm trên Drive hoặc file lớn. Kết quả được ghi vào `outputs/` trong repo Colab,
sau đó copy lại vào `ket_qua/` trên Drive để không mất khi Colab ngắt phiên.

In [ ]:
#@title 7a) Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

#@markdown Thư mục trên Drive chứa ảnh/video cần upscale:
DRIVE_FOLDER = "/content/drive/MyDrive/Upscale"  #@param {type:"string"}
import os
os.makedirs(DRIVE_FOLDER, exist_ok=True)
!ls -la "{DRIVE_FOLDER}" | head

In [ ]:
#@title 7b) Upscale cả thư mục Drive (ví dụ 4x, tile 400 chống hết VRAM)
!python -m valo_upscale.cli "{DRIVE_FOLDER}" -s 4 -o outputs --tile 400 --frame-format jpg
!mkdir -p "{DRIVE_FOLDER}/ket_qua" && cp outputs/*_upscaled_* "{DRIVE_FOLDER}/ket_qua/" && du -sh "{DRIVE_FOLDER}/ket_qua" && ls "{DRIVE_FOLDER}/ket_qua" | head

## 8) Giao diện web Gradio (tuỳ chọn)

Cài Gradio rồi mở UI có link công khai: kéo–thả file, chọn scale/model/engine, xem tiến trình
trực tiếp. Ô này chạy mãi cho tới khi bạn bấm **Stop** ở ô đó. Nếu Gradio cài xong mà Colab báo
lỗi `numpy`/`pydantic`, chọn *Runtime → Restart session* rồi chạy lại ô này (CLI không cần Gradio).

In [ ]:
#@title 8) Mở UI Gradio (link *.gradio.live)
!pip install -q -r requirements-ui.txt

import valo_upscale.app as app_module
demo = app_module.build_demo(out_dir="outputs")
demo.queue().launch(share=True, debug=False)

## 9) Mẹo & thông số

| Việc cần làm | Tham số |
|---|---|
| Phóng 2x / 3x / 4x / 6x / 8x | `-s 2`, `-s 3`, ... |
| Ảnh chụp, phong cảnh, vật thể | `-n realesrgan-x4plus` (mặc định cho ảnh) |
| Anime, manga, tranh 2D | `-n realesrgan-x4plus-anime` |
| Video (nhanh, giữ chuyển động) | `-n realesr-animevideov3` (mặc định cho video) |
| 2x "thuần AI" cho ảnh | `-n realesrgan-x2plus` |
| Hết VRAM | `--tile 400` hoặc `--tile 200` |
| Không cần AI (nhanh, CPU) | `-e ffmpeg_hq` |
| Chất lượng frame video cao nhất | `--frame-format png` |

* Video luôn bị clamp tối đa **3840×2160** (ngang) / **2160×3840** (dọc) — đúng như app gốc, để H.264 giải mã được.
* T4 16 GB: video 480p → 1080p dài ~10 s mất khoảng 30–90 s; ảnh 1000×1000 lên 4K mất vài giây.
* Colab free sẽ ngắt phiên khi đóng tab — tải kết quả về (ô 6c) hoặc lưu lên Drive (ô 7b) trước.
* Weight đã tải nằm ở `~/.cache/valo-upscale/weights`; muốn lưu lại giữa các phiên thì set
  `VALO_WEIGHTS_DIR` trỏ vào Drive.